# exp_adaptive_gate_v2 — retrieval-confidence routing, consistent metric (chase the oracle>SOTA lead)

v1 showed: a length gate routes by *format* (2022->rerank, 2023->retrieve) but text signals are too weak
for *within*-dataset routing — yet the **oracle gate on TREC22 = 0.617 > h2oloo 0.6125**, because the
ensemble actively hurts ~24% of TREC22 topics. If a label-free signal can flag those topics, routing them
to retrieval **beats SOTA on 2022**.

v2 fixes v1's metric bug (v1 mixed pytrec ensemble + ndcg_at_k retrieval) — **everything via pytrec here** —
and tests **retrieval-confidence** signals (top-score margin, BM25/dense agreement) instead of topic text.
Question: does the shape of the retrieval score distribution predict where reranking hurts? CPU-only, cached.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q datasets pytrec_eval

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
import numpy as np, pytrec_eval
from ctmatch.experiments import ExperimentConfig, load_eval
DATA_ROOT = '/content/drive/MyDrive/ct_data23'; T23 = f'{DATA_ROOT}/trec2023'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs')

def ndcg10(run, qrels):   # single, consistent metric (trec_eval semantics) for every ranking below
    ev = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut.10'})
    return {t: v['ndcg_cut_10'] for t, v in ev.evaluate(run).items()}

# ── TREC22 ──
s22 = load_eval(cfg, ['trec22'])['trec22']; rel22 = s22['rel_dict']; t2t22 = s22['topic2text']
pool22 = json.load(open(cfg.path('data/pool_nqs.json')))['trec22']
feat22 = {}
for l in open(cfg.path('data/retrieval_feats_nqs.jsonl')):
    r = json.loads(l)
    if r['source'] == 'trec22': feat22.setdefault(r['topic_id'], {})[r['doc_id']] = r
# ensemble ranking for 2022: reconstruct from booster + the cached feature files (so we get a RANKING, not
# just the cached per-topic NDCG) — needed for a consistent-metric ens and for rank-agreement signals.
import lightgbm as lgb
booster = lgb.Booster(model_file=cfg.path('models/ensemble_nqs.txt'))
FEAT = json.load(open(cfg.path('models/ensemble_nqs_features.json')))
def _load(name, key):
    d = {}
    p = cfg.feat_file(name)
    if os.path.exists(p):
        for l in open(p):
            r = json.loads(l)
            if r.get('source') == 'trec22': d[(r['topic_id'], r['doc_id'])] = r[key]
    return d
llm = _load('llm_scores', 'llm_score'); topi = _load('topicality', 'topicality')
cmf = cfg.feat_file('condition_match_exp');
cm = {}
if os.path.exists(cmf):
    for l in open(cmf):
        r = json.loads(l)
        if r.get('source') == 'trec22': cm[(r['topic_id'], r['doc_id'])] = r['condition_match']
ce = {}
for tag, path in [('clf', cfg.ce_cache_path('clf_R')), ('clft', cfg.ce_cache_path('clf_topic'))]:
    if os.path.exists(path): ce[tag] = np.load(path, allow_pickle=True)['d'].item()
def featvec(t, d):
    rf = feat22[t].get(d, {}); cr, cp = ce.get('clf', {}).get(('trec22', t, d), (0., 0.))
    tr, tp = ce.get('clft', {}).get(('trec22', t, d), (0., 0.))
    v = {'bm25': rf.get('bm25', 0.), 'bm25_rank': rf.get('bm25_rank', cfg.cand_k), 'dense': rf.get('dense', 0.),
         'dense_rank': rf.get('dense_rank', cfg.cand_k), 'rrf': rf.get('rrf', 0.), 'clf_rel': cr, 'clf_partial': cp,
         'clf_topic_rel': tr, 'clf_topic_partial': tp, 'llm_yesno': llm.get((t, d), cfg.llm_floor),
         'topicality': topi.get((t, d), 0.), 'condition_match': cm.get((t, d), 0.)}
    return [v[f] for f in FEAT]
ens_run22 = {}
for t in pool22:
    docs = [d for d in pool22[t] if t in feat22]
    if not docs: continue
    X = np.array([featvec(t, d) for d in docs], dtype=np.float32)
    ens_run22[t] = {d: float(s) for d, s in zip(docs, booster.predict(X))}
print('reconstructed TREC22 ensemble ranking for', len(ens_run22), 'topics')

# ── TREC23 ──
topics23 = {r['topic_id']: r['topic_text'] for r in map(json.loads, open(f'{T23}/topics2023_text.jsonl'))}
rel23 = {}
for l in open(f'{T23}/qrels2023.txt'):
    t, _, d, r = l.split(); rel23.setdefault(t, {})[d] = int(r)
topics23 = {t: x for t, x in topics23.items() if t in rel23}
pool23 = json.load(open(f'{T23}/pool_nqs_2023.json'))
feat23 = {}
for l in open(f'{T23}/retrieval_feats_2023.jsonl'):
    r = json.loads(l); feat23.setdefault(r['topic_id'], {})[r['doc_id']] = r
ens_run23 = {}
for l in open(f'{T23}/run_ext2023.txt'):
    t, _, d, rk, sc, _tag = l.split(); ens_run23.setdefault(t, {})[d] = float(sc)
print('TREC22', len(rel22), '| TREC23', len(topics23))

In [ ]:
# Consistent-metric ret/ens per topic + retrieval-confidence signals (all from cached retrieval scores).
def ret_run(feat_t, pool_t):
    return {t: {d: feat_t[t][d]['rrf'] for d in pool_t[t] if t in feat_t and d in feat_t[t]} for t in pool_t if t in feat_t}
def conf(feat_t_topic):
    docs = list(feat_t_topic)
    bm = sorted((feat_t_topic[d]['bm25'] for d in docs), reverse=True)
    dn = sorted((feat_t_topic[d]['dense'] for d in docs), reverse=True)
    def gap(s): return float(s[0] - s[9]) if len(s) >= 10 else float(s[0] - s[-1]) if len(s) > 1 else 0.0
    bmtop = set(sorted(docs, key=lambda d: feat_t_topic[d]['bm25'], reverse=True)[:20])
    dntop = set(sorted(docs, key=lambda d: feat_t_topic[d]['dense'], reverse=True)[:20])
    return {'bm25_gap': gap(bm), 'dense_gap': gap(dn), 'bm25_top1': float(bm[0]),
            'dense_top1': float(dn[0]), 'bm_dn_agree': len(bmtop & dntop) / 20.0}

rows = []
for ds, pool, feat, ensrun, rel, t2t in [
        ('TREC22', pool22, feat22, ens_run22, rel22, t2t22),
        ('TREC23', pool23, feat23, ens_run23, rel23, topics23)]:
    qrels = {t: {d: int(x) for d, x in rel[t].items()} for t in ensrun}
    nret = ndcg10(ret_run(feat, {t: pool[t] for t in ensrun}), qrels)
    nens = ndcg10(ensrun, qrels)
    for t in ensrun:
        if t not in feat: continue
        r = {'ds': ds, 't': t, 'ret': nret.get(t, 0), 'ens': nens.get(t, 0), 'len': len(t2t[t].split()), **conf(feat[t])}
        r['delta'] = r['ret'] - r['ens']; rows.append(r)
for ds in ['TREC22', 'TREC23']:
    R = [r for r in rows if r['ds'] == ds]
    print(f"{ds}: n={len(R)}  ret={np.mean([r['ret'] for r in R]):.4f}  ens={np.mean([r['ens'] for r in R]):.4f}  "
          f"prefer-retrieval={np.mean([r['delta']>0 for r in R]):.0%}")

In [ ]:
# Which signal predicts delta (ret-ens)? Focus on WITHIN-TREC22 routing — that is where the oracle>SOTA lives.
SIGS = ['len', 'bm25_gap', 'dense_gap', 'bm25_top1', 'dense_top1', 'bm_dn_agree']
print('correlation(signal, delta):        overall     within-TREC22   within-TREC23')
for s in SIGS:
    def c(R):
        x = np.array([r[s] for r in R]); y = np.array([r['delta'] for r in R])
        return np.corrcoef(x, y)[0, 1] if len(R) > 2 and x.std() > 0 else float('nan')
    r22 = [r for r in rows if r['ds'] == 'TREC22']; r23 = [r for r in rows if r['ds'] == 'TREC23']
    print(f'  {s:12s} {c(rows):>+12.3f} {c(r22):>+14.3f} {c(r23):>+14.3f}')

def per_ds(route_ret):
    out = {}
    for ds in ['TREC22', 'TREC23']:
        out[ds] = float(np.mean([(r['ret'] if route_ret[i] else r['ens']) for i, r in enumerate(rows) if r['ds'] == ds]))
    return out
always_ens = per_ds([False]*len(rows)); oracle = per_ds([r['delta'] > 0 for r in rows])
print(f"\n                  TREC22    TREC23")
print(f"always-ensemble  {always_ens['TREC22']:.4f}  {always_ens['TREC23']:.4f}   (current; consistent metric)")
print(f"ORACLE gate      {oracle['TREC22']:.4f}  {oracle['TREC23']:.4f}   (2022 vs h2oloo 0.6125)")

# Best single-signal threshold gate, and whether it captures the TREC22 headroom specifically.
best = None
for s in SIGS:
    for cut in sorted({r[s] for r in rows}):
        for direc in ['<', '>=']:
            route = [(r[s] < cut) if direc == '<' else (r[s] >= cut) for r in rows]
            g = per_ds(route); comb = g['TREC22'] + g['TREC23']
            if best is None or comb > best[0]: best = (comb, s, cut, direc, g)
_, s, cut, direc, g = best
print(f"\nbest gate: route to RETRIEVAL when {s} {direc} {cut:.4g}  ->  TREC22 {g['TREC22']:.4f}  TREC23 {g['TREC23']:.4f}")
print('If a retrieval-confidence signal drives TREC22 above 0.6125 (toward the 0.617 oracle) with a')
print('threshold set on a PRINCIPLE (not fit to labels), that is a clean, defensible SOTA-beat lead on 2022.')